# weight-decay-decoupled — faded example 2: Implement the AdamW path in a two-path comparison

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `weight-decay-decoupled`. Running the beacon reports progress on the `Optimizer: decoupled weight decay (AdamW)` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: decoupled weight decay (AdamW)` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`weight-decay-decoupled`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "weight-decay-decoupled"
DD_SUBTOPIC = "Optimizer: decoupled weight decay (AdamW)"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

To understand what AdamW adds over Adam, you can implement both side-by-side and observe their divergence. In the AdamW path, `p * (1 - lr*wd)` reduces the parameter before the gradient step. In the Adam+L2 path, `wd * p` is added to the gradient before the moment update. The two are equivalent only when gradients are rescaled by a constant — not in practice with Adam's adaptive normalization.

## Faded exercise 2

Implement the AdamW path in `two_path_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step)`. The Adam+L2 (coupled) path is already implemented. Complete the blank that implements the AdamW (decoupled) path and returns both updated parameter values.

Return `(p_adamw, p_l2)` — both as new tensors (do not mutate the inputs).

**Fill in:** Implement the AdamW path: clone p, apply p_aw.mul_(1 - lr*wd), run Adam moment update on raw grad, apply bias correction, then p_aw.addcdiv_. Return (p_aw, p_l2).

In [ ]:
import torch as t

def two_path_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    # Adam+L2 path
    p_l2 = p.clone()
    g_eff = grad + wd * p_l2
    m_l2 = beta1 * m + (1 - beta1) * g_eff
    v_l2 = beta2 * v + (1 - beta2) * g_eff**2
    m_hat_l2 = m_l2 / (1 - beta1**step)
    v_hat_l2 = v_l2 / (1 - beta2**step)
    p_l2 = p_l2 - lr * m_hat_l2 / (v_hat_l2.sqrt() + eps)

    # AdamW path
    p_aw = p.clone()
    p_aw.mul_(1 - lr * wd)
    m_aw = beta1 * m + (1 - beta1) * grad
    v_aw = beta2 * v + (1 - beta2) * grad**2
    m_hat_aw = m_aw / (1 - beta1**step)
    v_hat_aw = v_aw / (1 - beta2**step)
    p_aw.addcdiv_(m_hat_aw, v_hat_aw.sqrt().add_(eps), value=-lr)

    return p_aw, p_l2

t.manual_seed(3)
p = t.tensor([0.7])
g = t.tensor([0.15])
m = t.zeros(1); v = t.zeros(1)
p_aw, p_l2 = two_path_step(p, g, m, v, 5e-3, 0.9, 0.999, 1e-8, 0.05, 1)
print('AdamW p:', p_aw.item(), '| L2 p:', p_l2.item(), '| differ:', not t.allclose(p_aw, p_l2))


import torch as t

def two_path_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    p_l2 = p.clone()
    g_eff = grad + wd * p_l2
    m_l2 = beta1 * m + (1 - beta1) * g_eff
    v_l2 = beta2 * v + (1 - beta2) * g_eff**2
    m_hat_l2 = m_l2 / (1 - beta1**step)
    v_hat_l2 = v_l2 / (1 - beta2**step)
    p_l2 = p_l2 - lr * m_hat_l2 / (v_hat_l2.sqrt() + eps)

    p_aw = p.clone()
    p_aw.mul_(1 - lr * wd)
    m_aw = beta1 * m + (1 - beta1) * grad
    v_aw = beta2 * v + (1 - beta2) * grad**2
    m_hat_aw = m_aw / (1 - beta1**step)
    v_hat_aw = v_aw / (1 - beta2**step)
    p_aw.addcdiv_(m_hat_aw, v_hat_aw.sqrt().add_(eps), value=-lr)
    return p_aw, p_l2

def _test():
    t.manual_seed(0)
    p = t.tensor([0.5])
    g = t.tensor([0.1])
    m = t.zeros(1); v = t.zeros(1)
    lr, b1, b2, eps, wd = 1e-3, 0.9, 0.999, 1e-8, 0.01
    p_aw, p_l2 = two_path_step(p, g, m, v, lr, b1, b2, eps, wd, 1)
    # AdamW must match torch.optim.AdamW
    ref_p = t.tensor([0.5], requires_grad=True)
    opt = t.optim.AdamW([ref_p], lr=lr, betas=(b1, b2), eps=eps, weight_decay=wd)
    opt.zero_grad()
    ref_p.grad = t.tensor([0.1])
    opt.step()
    assert t.allclose(p_aw, ref_p.detach(), atol=1e-6), f'{p_aw} != {ref_p}'
    assert not t.allclose(p_aw, p_l2), 'AdamW and L2 should differ with wd>0'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def two_path_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    # Adam+L2 path
    p_l2 = p.clone()
    g_eff = grad + wd * p_l2
    m_l2 = beta1 * m + (1 - beta1) * g_eff
    v_l2 = beta2 * v + (1 - beta2) * g_eff**2
    m_hat_l2 = m_l2 / (1 - beta1**step)
    v_hat_l2 = v_l2 / (1 - beta2**step)
    p_l2 = p_l2 - lr * m_hat_l2 / (v_hat_l2.sqrt() + eps)

    # AdamW path
    p_aw = p.clone()
    p_aw.mul_(1 - lr * wd)
    m_aw = beta1 * m + (1 - beta1) * grad
    v_aw = beta2 * v + (1 - beta2) * grad**2
    m_hat_aw = m_aw / (1 - beta1**step)
    v_hat_aw = v_aw / (1 - beta2**step)
    p_aw.addcdiv_(m_hat_aw, v_hat_aw.sqrt().add_(eps), value=-lr)

    return p_aw, p_l2

t.manual_seed(3)
p = t.tensor([0.7])
g = t.tensor([0.15])
m = t.zeros(1); v = t.zeros(1)
p_aw, p_l2 = two_path_step(p, g, m, v, 5e-3, 0.9, 0.999, 1e-8, 0.05, 1)
print('AdamW p:', p_aw.item(), '| L2 p:', p_l2.item(), '| differ:', not t.allclose(p_aw, p_l2))
```
</details>